# Hashtag Generation from Image Features

This notebook trains a lightweight neural hashtag-generation model using pre-extracted image features and associated hashtag text.

## Pipeline

1. Load pre-extracted image features.
2. Load and clean hashtag annotations.
3. Build a hashtag vocabulary.
4. Encode and pad hashtag sequences.
5. Train a GRU-based generative model conditioned on image features.
6. Generate hashtags for test samples.

> **Data note:** The original notebook used a dataset archive stored in Google Drive. This cleaned version uses a local `data/` directory so that the notebook is not tied to a particular machine or Colab account.


In [ ]:
from pathlib import Path
import os
import re
from collections import Counter

# Project paths
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "artifacts"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected input files
FEATURES_FILE = DATA_DIR / "harrison_features.npz"
HASHTAGS_FILE = DATA_DIR / "tag_list.txt"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Artifacts directory:", OUTPUT_DIR)


## 1. Load Dependencies

The model is implemented in PyTorch. Pandas and NumPy are used for data preparation, while scikit-learn and tqdm support dataset handling and progress reporting.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Load Pre-extracted Image Features

The project uses the `imagenet_fc_layers` feature representation stored in the NPZ file.

Place the required files in:

```text
data/
├── harrison_features.npz
└── tag_list.txt
```

The dataset files are not included in this repository if they are too large or subject to distribution restrictions.


In [ ]:
if not FEATURES_FILE.exists():
    raise FileNotFoundError(
        f"Missing feature file: {FEATURES_FILE}. "
        "Place harrison_features.npz inside the data/ directory."
    )

with np.load(FEATURES_FILE) as data:
    if "imagenet_fc_layers" not in data.files:
        raise KeyError(
            "The NPZ file does not contain the expected 'imagenet_fc_layers' array. "
            f"Available keys: {data.files}"
        )
    image_features = data["imagenet_fc_layers"]

print("Image feature shape:", image_features.shape)


In [ ]:
if not HASHTAGS_FILE.exists():
    raise FileNotFoundError(
        f"Missing hashtag file: {HASHTAGS_FILE}. "
        "Place tag_list.txt inside the data/ directory."
    )

with HASHTAGS_FILE.open("r", encoding="utf-8") as f:
    hashtags_list = [line.strip() for line in f if line.strip()]

print(f"Number of hashtag annotations: {len(hashtags_list)}")
print("First 5 annotations:", hashtags_list[:5])


## 3. Combine and Clean the Data

Image features and hashtag annotations are combined into a single DataFrame. The original experiment limited the training subset to the first 10,000 samples to reduce training time; that experimental choice is retained here.


In [ ]:
if len(image_features) != len(hashtags_list):
    raise ValueError(
        "The number of image features and hashtag annotations must match. "
        f"Got {len(image_features)} features and {len(hashtags_list)} annotations."
    )

df_combined = pd.DataFrame({
    "features": list(image_features),
    "hashtags": hashtags_list,
})

MAX_SAMPLES = min(10_000, len(df_combined))
df = df_combined.head(MAX_SAMPLES).copy()

print("Full dataset shape:", df_combined.shape)
print("Training subset shape:", df.shape)


In [ ]:
def clean_hashtags(text: str) -> str:
    text = text.lower()
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()

df["hashtags"] = df["hashtags"].apply(clean_hashtags)

# Convert serialized feature vectors to NumPy arrays if necessary.
if isinstance(df.iloc[0]["features"], str):
    df["features"] = df["features"].apply(
        lambda x: np.fromstring(x.strip("[]"), sep=",")
    )


## 4. Build the Hashtag Vocabulary

Hashtags are tokenized and mapped to integer IDs. Special tokens are used for padding, sequence boundaries, and unknown words.

The original experiment retained the top 5,000 hashtag tokens.


In [ ]:
TOP_K = 5000

hashtags_tokenized = [tags.split() for tags in df["hashtags"]]
all_tags = [tag for tags in hashtags_tokenized for tag in tags]

counter = Counter(all_tags)
most_common = counter.most_common(TOP_K)

vocab = ["<pad>", "<start>", "<end>", "<unk>"] + [
    word for word, _ in most_common
]

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

print("Vocabulary size:", len(vocab))


In [ ]:
def encode_tags(tags):
    ids = [word2idx.get(word, word2idx["<unk>"]) for word in tags]
    return [word2idx["<start>"]] + ids + [word2idx["<end>"]]

def pad_sequence(seq, max_len=10):
    seq = seq[:max_len]
    return seq + [word2idx["<pad>"]] * (max_len - len(seq))

df["hashtags_split"] = df["hashtags"].apply(lambda x: x.split())
df["encoded"] = df["hashtags_split"].apply(encode_tags)
df["padded"] = df["encoded"].apply(pad_sequence)

MAX_TAG_LENGTH = 10


## 5. Dataset and DataLoader

The custom PyTorch dataset converts the pre-extracted image features and encoded hashtag sequences into tensors.


In [ ]:
class HashtagDataset(Dataset):
    def __init__(self, dataframe):
        self.X = np.stack(dataframe["features"].to_numpy())
        self.y = np.stack(dataframe["padded"].to_numpy())

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        feature = torch.tensor(self.X[idx], dtype=torch.float32)
        sequence = torch.tensor(self.y[idx], dtype=torch.long)
        return feature, sequence


train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
)

train_dataset = HashtagDataset(train_df)
val_dataset = HashtagDataset(val_df)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


## 6. Generative Hashtag Model

The model projects the image feature vector into an embedding space, combines it with learned hashtag-token embeddings, and uses a GRU to generate the hashtag sequence.


In [ ]:
class TinyHashtagGenerator(nn.Module):
    def __init__(
        self,
        feature_dim,
        vocab_size,
        embed_dim=128,
        hidden_dim=128,
    ):
        super().__init__()

        self.feature_proj = nn.Linear(feature_dim, embed_dim)
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, sequences):
        feature_embedding = self.feature_proj(features).unsqueeze(1)
        token_embeddings = self.embed(sequences)

        # Condition sequence generation on the image feature.
        gru_input = torch.cat(
            [feature_embedding, token_embeddings[:, :-1, :]],
            dim=1,
        )

        output, _ = self.gru(gru_input)
        return self.fc(output)


FEATURE_DIM = train_dataset.X.shape[1]
VOCAB_SIZE = len(vocab)

model = TinyHashtagGenerator(
    feature_dim=FEATURE_DIM,
    vocab_size=VOCAB_SIZE,
).to(device)

print(model)


## 7. Training

The original experiment uses cross-entropy loss with padding tokens ignored and the Adam optimizer. The training configuration is retained from the original implementation.


In [ ]:
criterion = nn.CrossEntropyLoss(
    ignore_index=word2idx["<pad>"]
)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
)

EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for features, sequences in tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    ):
        features = features.to(device)
        sequences = sequences.to(device)

        logits = model(features, sequences)

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            sequences.reshape(-1),
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / max(len(train_loader), 1)
    print(f"Epoch {epoch + 1}: loss = {average_loss:.4f}")


## 8. Save Model Weights

The trained model weights are saved outside the source notebook. Large model artifacts should generally not be committed to GitHub unless there is a specific reason to do so.


In [ ]:
MODEL_PATH = OUTPUT_DIR / "tiny_hashtag_generator.pth"

torch.save(model.state_dict(), MODEL_PATH)
print(f"Model weights saved to: {MODEL_PATH}")


## 9. Hashtag Generation

The trained model generates hashtags autoregressively from an image feature vector.


In [ ]:
def generate_hashtags(model, feature, max_len=5, temperature=1.0):
    model.eval()

    with torch.no_grad():
        feature_tensor = (
            torch.tensor(feature, dtype=torch.float32)
            .unsqueeze(0)
            .to(device)
        )

        feature_embedding = model.feature_proj(feature_tensor).unsqueeze(1)

        generated = [
            word2idx["<start>"]
        ]

        hidden = None

        for _ in range(max_len):
            sequence_tensor = torch.tensor(
                generated,
                dtype=torch.long,
                device=device,
            ).unsqueeze(0)

            token_embeddings = model.embed(sequence_tensor)

            gru_input = torch.cat(
                [feature_embedding, token_embeddings],
                dim=1,
            )

            output, hidden = model.gru(
                gru_input,
                hidden,
            )

            logits = model.fc(output[:, -1, :])

            if temperature > 0:
                logits = logits / temperature

            next_token = torch.argmax(logits, dim=-1).item()

            if next_token == word2idx["<end>"]:
                break

            generated.append(next_token)

        return [
            idx2word[token]
            for token in generated[1:]
            if token not in {
                word2idx["<pad>"],
                word2idx["<end>"],
            }
        ]


## 10. Test the Generator

The following cells compare generated hashtags with the original annotations for selected samples.


In [ ]:
sample_idx = 0

sample_feature = df.iloc[sample_idx]["features"]
predicted_tags = generate_hashtags(
    model,
    sample_feature,
)

print("Generated hashtags:", predicted_tags)
print("Original hashtags:", df.iloc[sample_idx]["hashtags"])


In [ ]:
# Compare a few samples from the full combined dataset.
start_idx = 10200
num_samples = 5

end_idx = min(start_idx + num_samples, len(df_combined))

for i in range(start_idx, end_idx):
    sample_feature = df_combined.iloc[i]["features"]
    predicted_tags = generate_hashtags(model, sample_feature)

    print(f"\n--- Sample {i + 1} ---")
    print("Generated hashtags:", predicted_tags)
    print("Original hashtags:", df_combined.iloc[i]["hashtags"])


## Notes on Reproducibility

- Place the required feature and hashtag files in `data/`.
- Run the notebook from the project root so the relative paths resolve correctly.
- The notebook uses a fixed `random_state=42` for the train/validation split.
- Training the model requires PyTorch and the dependencies listed in the project's `requirements.txt`.
- The original experiment's 10,000-sample training limit and 30-epoch configuration are retained.
- Model weights are saved under `artifacts/` and are not required to be committed to the repository.
